# VespAI Main Application (main.py)
This notebook contains the actual code from `main.py` split into logical notebook cells for direct use and customization.

# Convert main.py to Jupyter Notebook
This notebook demonstrates how to convert the `main.py` file from the VespAI project into a Jupyter notebook, splitting the code into logical sections and saving the result in the `notebooks` folder.

## 1. Read main.py Source Code
Read the contents of the `main.py` file using Python's file I/O operations.

In [ ]:
# Read the contents of main.py
main_py_path = '../src/vespai/main.py'
with open(main_py_path, 'r') as f:
    main_source = f.read()
print(main_source[:500])  # Preview the first 500 characters

## 2. Split Code into Notebook Cells
Process the source code to split it into logical notebook cells, such as by function or class definitions, or by comments indicating sections.

In [ ]:
# Example: Split by top-level class, function, and import blocks
import re

def split_code_to_cells(source):
    # Split on class, def, or import blocks for demonstration
    pattern = r'(^class\s+|^def\s+|^from\s+|^import\s+|^if __name__ == \'__main__\':)'  # crude split
    parts = re.split(pattern, source, flags=re.MULTILINE)
    # Re-attach split tokens
    cells = []
    i = 0
    while i < len(parts):
        if i == 0:
            cells.append(parts[i])
            i += 1
        else:
            cell = parts[i] + parts[i+1]
            cells.append(cell)
            i += 2
    return [c.strip() for c in cells if c.strip()]

code_cells = split_code_to_cells(main_source)
print(f"Split into {len(code_cells)} logical cells.")
for idx, cell in enumerate(code_cells[:3]):
    print(f"\n--- Cell {idx+1} ---\n{cell[:300]}")

## 3. Create Jupyter Notebook Object
Use the nbformat library to create a new Jupyter notebook object and populate it with the extracted code cells.

In [ ]:
import nbformat
from nbformat.v4 import new_notebook, new_code_cell

# Create a new notebook object
nb = new_notebook()
nb.cells = [new_code_cell(cell) for cell in code_cells]

# Preview the first cell
print(nb.cells[0].source[:300])

## 4. Save Notebook File to notebooks Folder
Write the notebook object to a `.ipynb` file under the `notebooks` directory using nbformat's write function.

In [ ]:
# Save the notebook to the notebooks folder
notebook_path = 'main_converted.ipynb'  # Save as a new file to avoid overwriting this notebook
with open(notebook_path, 'w') as f:
    nbformat.write(nb, f)
print(f"Notebook saved to {notebook_path}")

In [35]:
#!/usr/bin/env python3
"""
VespAI Main Application

Modular main application that coordinates all VespAI components for hornet detection.
This replaces the monolithic web_preview.py with a clean, testable architecture.

Author: Jakob Zeise (Zeise Digital)
Version: 1.0
"""

import logging
import sys
import time
import threading
import signal
from typing import Optional

In [36]:
# Core modules
from pathlib import Path

# Ensure project root is on sys.path so `src.vespai...` imports work in notebooks
project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.vespai.core.config import create_config_from_args
from src.vespai.core.detection import CameraManager, ModelManager, DetectionProcessor
from src.vespai.sms.lox24 import create_sms_manager_from_env
from src.vespai.web.routes import register_routes

# External dependencies
try:
    from flask import Flask
    import cv2
    import torch
except ImportError as e:
    print(f"Missing required dependency: {e}")
    print("Please install dependencies with: pip install -r requirements.txt")
    sys.exit(1)

In [37]:
# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


In [38]:
class VespAIApplication:
    """
    Main VespAI application that orchestrates all components.
    Provides a clean, modular architecture for hornet detection with
    camera management, model inference, web interface, and SMS alerts.
    """
    def __init__(self):
        """Initialize the VespAI application."""
        self.config = None
        self.camera_manager = None
        self.model_manager = None
        self.detection_processor = None
        self.sms_manager = None
        self.flask_app = None
        self.web_thread = None
        self.running = False
        self.web_frame = None
        self.web_lock = threading.Lock()
        signal.signal(signal.SIGINT, self._signal_handler)
        signal.signal(signal.SIGTERM, self._signal_handler)
# ...methods are now split into separate cells...

In [39]:
    def initialize(self, args=None):
        """
        Initialize all application components.
        Args:
            args: Command line arguments (None for sys.argv)
        """
        logger.info("Initializing VespAI application...")
        # Load configuration
        self.config = create_config_from_args(args)
        self.config.print_summary()
        # Initialize components
        self._initialize_camera()
        self._initialize_model()
        self._initialize_detection_processor()
        self._initialize_sms()
        if self.config.get('enable_web'):
            self._initialize_web_interface()
        logger.info("VespAI application initialized successfully")

In [40]:
    def _initialize_camera(self):
        logger.info("Initializing camera...")
        resolution = self.config.get_camera_resolution()
        self.camera_manager = CameraManager(resolution)
        video_file = self.config.get('video_file')
        self.camera_manager.initialize_camera(video_file)

    def _initialize_model(self):
        logger.info("Initializing detection model...")
        model_path = self.config.get('model_path')
        confidence = self.config.get('confidence_threshold')
        self.model_manager = ModelManager(model_path, confidence)
        self.model_manager.load_model()

    def _initialize_detection_processor(self):
        logger.info("Initializing detection processor...")
        self.detection_processor = DetectionProcessor()

    def _initialize_sms(self):
        if self.config.get('enable_sms'):
            logger.info("Initializing SMS alerts...")
            self.sms_manager = create_sms_manager_from_env()
            if self.sms_manager:
                logger.info("SMS alerts enabled")
            else:
                logger.warning("SMS configuration incomplete - alerts disabled")
        else:
            logger.info("SMS alerts disabled")

In [41]:
    def _initialize_web_interface(self):
        logger.info("Initializing web interface...")
        import os
        web_dir = os.path.join(os.path.dirname(__file__), 'web')
        template_dir = os.path.join(web_dir, 'templates')
        static_dir = os.path.join(web_dir, 'static')
        self.flask_app = Flask(__name__, 
                              template_folder=template_dir,
                              static_folder=static_dir,
                              static_url_path='/static')
        register_routes(
            self.flask_app,
            self.detection_processor.stats,
            self.detection_processor.hourly_detections,
            self
        )
        web_config = self.config.get_web_config()
        self.web_thread = threading.Thread(
            target=self._run_web_server,
            args=(web_config['host'], web_config['port']),
            daemon=True
        )
        self.web_thread.start()
        time.sleep(0.5)
        logger.info("Web interface starting at %s", web_config['public_url'])

    def _run_web_server(self, host: str, port: int):
        try:
            self.flask_app.run(host=host, port=port, threaded=True, debug=False)
        except Exception as e:
            logger.error("Web server error: %s", e)

In [42]:
    def run(self):
        if not self._validate_initialization():
            logger.error("Application not properly initialized")
            return False
        logger.info("Starting VespAI detection system...")
        logger.info("Press Ctrl+C to stop")
        self.running = True
        frame_count = 0
        fps_start_time = time.time()
        fps_counter = 0
        last_frame_time = time.time()
        last_stats_update = time.time()
        try:
            while self.running:
                loop_start = time.time()
                current_time = time.time()
                if current_time - last_frame_time > 30:
                    logger.warning("System appears to be hanging - attempting recovery...")
                    self._attempt_recovery()
                    last_frame_time = current_time
                try:
                    success, frame = self.camera_manager.read_frame()
                    if not success or frame is None:
                        logger.warning("Failed to read frame, retrying...")
                        time.sleep(0.1)
                        continue
                    last_frame_time = current_time
                except Exception as e:
                    logger.error(f"Camera error: {e}")
                    time.sleep(1)
                    continue
                frame_count += 1
                fps_counter += 1
                self.detection_processor.stats['frame_id'] = frame_count
                if frame_count % 30 == 0:
                    logger.debug(f"Frame count updated: {frame_count}")
                if time.time() - fps_start_time >= 1.0:
                    self.detection_processor.stats['fps'] = fps_counter
                    fps_counter = 0
                    fps_start_time = time.time()
                try:
                    results = self.model_manager.predict(frame)
                    velutina_count, crabro_count, annotated_frame = self.detection_processor.process_detections(
                        results, frame, frame_count, self.config.get('confidence_threshold')
                    )
                except Exception as e:
                    logger.error(f"Detection error: {e}")
                    velutina_count = crabro_count = 0
                    annotated_frame = frame.copy()
                if velutina_count > 0 or crabro_count > 0:
                    self._handle_detection(velutina_count, crabro_count, frame_count, annotated_frame)
                if self.config.get('enable_web'):
                    try:
                        display_frame = cv2.resize(annotated_frame, (640, 360))
                        with self.web_lock:
                            self.web_frame = display_frame.copy()
                    except Exception as e:
                        logger.error(f"Web frame update error: {e}")
                if current_time - last_stats_update > 10:
                    self.detection_processor.stats['last_update'] = current_time
                    last_stats_update = current_time
                if self.config.get('print_detections') and (velutina_count > 0 or crabro_count > 0):
                    confidence = self.detection_processor.stats.get('confidence_avg', 0)
                    print(f"Frame {frame_count}: {velutina_count} Velutina, {crabro_count} Crabro "
                          f"(confidence: {confidence:.1f}%)")
                frame_delay = self.config.get('frame_delay', 0.3)
                elapsed = time.time() - loop_start
                if elapsed < frame_delay:
                    time.sleep(frame_delay - elapsed)
        except KeyboardInterrupt:
            logger.info("Received interrupt signal")
            self.running = False
        except Exception as e:
            logger.error("Unexpected error in detection loop: %s", e)
            self.running = False
            return False
        finally:
            self._cleanup()
        logger.info("VespAI detection system stopped")
        return True

In [43]:
    def _handle_detection(self, velutina_count: int, crabro_count: int, frame_id: int, frame):
        if self.config.get('save_detections'):
            self._save_detection_image(frame, frame_id, velutina_count, crabro_count)
        if self.sms_manager:
            self._send_sms_alert(velutina_count, crabro_count, frame_id)

    def _save_detection_image(self, frame, frame_id: int, velutina: int, crabro: int):
        import os
        from datetime import datetime
        save_dir = self.config.get('save_directory', 'data/detections')
        os.makedirs(save_dir, exist_ok=True)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        species = 'velutina' if velutina > 0 else 'crabro'
        filename = f"{timestamp}_frame{frame_id}_{species}_{velutina}v_{crabro}c.jpg"
        filepath = os.path.join(save_dir, filename)
        cv2.imwrite(filepath, frame)
        logger.info("Saved detection image: %s", filepath)

    def _send_sms_alert(self, velutina_count: int, crabro_count: int, frame_id: int):
        if not self.sms_manager:
            return
        web_config = self.config.get_web_config()
        current_time = time.strftime('%H%M%S')
        detection_key = f"{frame_id}_{current_time}"
        frame_url = f"{web_config['public_url']}/frame/{detection_key}"
        if velutina_count > 0:
            hornet_type = 'velutina'
            count = velutina_count
        else:
            hornet_type = 'crabro'
            count = crabro_count
        confidence = self.detection_processor.stats.get('confidence_avg', 0)
        message = self.sms_manager.create_hornet_alert(hornet_type, count, confidence, frame_url)
        success, status = self.sms_manager.send_alert(message)
        if success:
            logger.info("SMS alert sent: %s", status)
        else:
            logger.warning("SMS alert failed: %s", status)

In [44]:
    def _validate_initialization(self) -> bool:
        if not self.camera_manager:
            logger.error("Camera manager not initialized")
            return False
        if not self.model_manager or not self.model_manager.model:
            logger.error("Model manager not initialized") 
            return False
        if not self.detection_processor:
            logger.error("Detection processor not initialized")
            return False
        return True

    def _signal_handler(self, signum, frame):
        logger.info("Received signal %d, shutting down...", signum)
        self.running = False
        if hasattr(self, '_shutdown_requested'):
            logger.info("Force shutdown requested, terminating immediately...")
            import os
            os._exit(0)
        self._shutdown_requested = True

    def _attempt_recovery(self):
        logger.info("Attempting system recovery...")
        try:
            import gc
            gc.collect()
            if self.camera_manager:
                logger.info("Resetting camera connection...")
                self.camera_manager.release()
                time.sleep(2)
                self.camera_manager.initialize_camera()
            with self.web_lock:
                self.web_frame = None
            logger.info("Recovery attempt completed")
        except Exception as e:
            logger.error(f"Recovery failed: {e}")

    def _cleanup(self):
        logger.info("Cleaning up resources...")
        if self.camera_manager:
            self.camera_manager.release()
        if self.detection_processor:
            stats = self.detection_processor.stats
            logger.info("Final statistics:")
            logger.info("  Total frames processed: %d", stats.get('frame_id', 0))
            logger.info("  Total detections: %d", stats.get('total_detections', 0))
            logger.info("  Asian hornets: %d", stats.get('total_velutina', 0))
            logger.info("  European hornets: %d", stats.get('total_crabro', 0))

In [45]:
def main():
    """Main entry point for the VespAI application."""
    app = VespAIApplication()
    try:
        app.initialize()
        success = app.run()
        sys.exit(0 if success else 1)
    except Exception as e:
        logger.error("Fatal error: %s", e)
        sys.exit(1)

if __name__ == '__main__':
    main()

AttributeError: 'VespAIApplication' object has no attribute '_signal_handler'

In [46]:
def initialize(self, args=None):
    """
    Initialize all application components.
    Args:
        args: Command line arguments (None for sys.argv)
    """
    logger.info("Initializing VespAI application...")
    self.config = create_config_from_args(args)
    self.config.print_summary()
    self._initialize_camera()
    self._initialize_model()
    self._initialize_detection_processor()
    self._initialize_sms()
    if self.config.get('enable_web'):
        self._initialize_web_interface()
    logger.info("VespAI application initialized successfully")

In [47]:
def _initialize_camera(self):
    logger.info("Initializing camera...")
    resolution = self.config.get_camera_resolution()
    self.camera_manager = CameraManager(resolution)
    video_file = self.config.get('video_file')
    self.camera_manager.initialize_camera(video_file)

In [48]:
def _initialize_model(self):
    logger.info("Initializing detection model...")
    model_path = self.config.get('model_path')
    confidence = self.config.get('confidence_threshold')
    self.model_manager = ModelManager(model_path, confidence)
    self.model_manager.load_model()

In [49]:
def _initialize_detection_processor(self):
    logger.info("Initializing detection processor...")
    self.detection_processor = DetectionProcessor()

In [50]:
def _initialize_sms(self):
    if self.config.get('enable_sms'):
        logger.info("Initializing SMS alerts...")
        self.sms_manager = create_sms_manager_from_env()
        if self.sms_manager:
            logger.info("SMS alerts enabled")
        else:
            logger.warning("SMS configuration incomplete - alerts disabled")
    else:
        logger.info("SMS alerts disabled")

In [51]:
def _initialize_web_interface(self):
    logger.info("Initializing web interface...")
    import os
    web_dir = os.path.join(os.path.dirname(__file__), 'web')
    template_dir = os.path.join(web_dir, 'templates')
    static_dir = os.path.join(web_dir, 'static')
    self.flask_app = Flask(__name__, 
                          template_folder=template_dir,
                          static_folder=static_dir,
                          static_url_path='/static')
    register_routes(
        self.flask_app,
        self.detection_processor.stats,
        self.detection_processor.hourly_detections,
        self
    )
    web_config = self.config.get_web_config()
    self.web_thread = threading.Thread(
        target=self._run_web_server,
        args=(web_config['host'], web_config['port']),
        daemon=True
    )
    self.web_thread.start()
    time.sleep(0.5)
    logger.info("Web interface starting at %s", web_config['public_url'])

In [52]:
def _run_web_server(self, host: str, port: int):
    try:
        self.flask_app.run(host=host, port=port, threaded=True, debug=False)
    except Exception as e:
        logger.error("Web server error: %s", e)

In [53]:
def run(self):
    if not self._validate_initialization():
        logger.error("Application not properly initialized")
        return False
    logger.info("Starting VespAI detection system...")
    logger.info("Press Ctrl+C to stop")
    self.running = True
    frame_count = 0
    fps_start_time = time.time()
    fps_counter = 0
    last_frame_time = time.time()
    last_stats_update = time.time()
    try:
        while self.running:
            loop_start = time.time()
            current_time = time.time()
            if current_time - last_frame_time > 30:
                logger.warning("System appears to be hanging - attempting recovery...")
                self._attempt_recovery()
                last_frame_time = current_time
            try:
                success, frame = self.camera_manager.read_frame()
                if not success or frame is None:
                    logger.warning("Failed to read frame, retrying...")
                    time.sleep(0.1)
                    continue
                last_frame_time = current_time
            except Exception as e:
                logger.error(f"Camera error: {e}")
                time.sleep(1)
                continue
            frame_count += 1
            fps_counter += 1
            self.detection_processor.stats['frame_id'] = frame_count
            if frame_count % 30 == 0:
                logger.debug(f"Frame count updated: {frame_count}")
            if time.time() - fps_start_time >= 1.0:
                self.detection_processor.stats['fps'] = fps_counter
                fps_counter = 0
                fps_start_time = time.time()
            try:
                results = self.model_manager.predict(frame)
                velutina_count, crabro_count, annotated_frame = self.detection_processor.process_detections(
                    results, frame, frame_count, self.config.get('confidence_threshold')
                )
            except Exception as e:
                logger.error(f"Detection error: {e}")
                velutina_count = crabro_count = 0
                annotated_frame = frame.copy()
            if velutina_count > 0 or crabro_count > 0:
                self._handle_detection(velutina_count, crabro_count, frame_count, annotated_frame)
            if self.config.get('enable_web'):
                try:
                    display_frame = cv2.resize(annotated_frame, (640, 360))
                    with self.web_lock:
                        self.web_frame = display_frame.copy()
                except Exception as e:
                    logger.error(f"Web frame update error: {e}")
            if current_time - last_stats_update > 10:
                self.detection_processor.stats['last_update'] = current_time
                last_stats_update = current_time
            if self.config.get('print_detections') and (velutina_count > 0 or crabro_count > 0):
                confidence = self.detection_processor.stats.get('confidence_avg', 0)
                print(f"Frame {frame_count}: {velutina_count} Velutina, {crabro_count} Crabro "
                      f"(confidence: {confidence:.1f}%)")
            frame_delay = self.config.get('frame_delay', 0.3)
            elapsed = time.time() - loop_start
            if elapsed < frame_delay:
                time.sleep(frame_delay - elapsed)
    except KeyboardInterrupt:
        logger.info("Received interrupt signal")
        self.running = False
    except Exception as e:
        logger.error("Unexpected error in detection loop: %s", e)
        self.running = False
        return False
    finally:
        self._cleanup()
    logger.info("VespAI detection system stopped")
    return True

In [54]:
def _handle_detection(self, velutina_count: int, crabro_count: int, frame_id: int, frame):
    if self.config.get('save_detections'):
        self._save_detection_image(frame, frame_id, velutina_count, crabro_count)
    if self.sms_manager:
        self._send_sms_alert(velutina_count, crabro_count, frame_id)

In [55]:
def _save_detection_image(self, frame, frame_id: int, velutina: int, crabro: int):
    import os
    from datetime import datetime
    save_dir = self.config.get('save_directory', 'data/detections')
    os.makedirs(save_dir, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    species = 'velutina' if velutina > 0 else 'crabro'
    filename = f"{timestamp}_frame{frame_id}_{species}_{velutina}v_{crabro}c.jpg"
    filepath = os.path.join(save_dir, filename)
    cv2.imwrite(filepath, frame)
    logger.info("Saved detection image: %s", filepath)

In [56]:
def _send_sms_alert(self, velutina_count: int, crabro_count: int, frame_id: int):
    if not self.sms_manager:
        return
    web_config = self.config.get_web_config()
    current_time = time.strftime('%H%M%S')
    detection_key = f"{frame_id}_{current_time}"
    frame_url = f"{web_config['public_url']}/frame/{detection_key}"
    if velutina_count > 0:
        hornet_type = 'velutina'
        count = velutina_count
    else:
        hornet_type = 'crabro'
        count = crabro_count
    confidence = self.detection_processor.stats.get('confidence_avg', 0)
    message = self.sms_manager.create_hornet_alert(hornet_type, count, confidence, frame_url)
    success, status = self.sms_manager.send_alert(message)
    if success:
        logger.info("SMS alert sent: %s", status)
    else:
        logger.warning("SMS alert failed: %s", status)

In [57]:
def _validate_initialization(self) -> bool:
    if not self.camera_manager:
        logger.error("Camera manager not initialized")
        return False
    if not self.model_manager or not self.model_manager.model:
        logger.error("Model manager not initialized") 
        return False
    if not self.detection_processor:
        logger.error("Detection processor not initialized")
        return False
    return True

In [58]:
def _signal_handler(self, signum, frame):
    logger.info("Received signal %d, shutting down...", signum)
    self.running = False
    if hasattr(self, '_shutdown_requested'):
        logger.info("Force shutdown requested, terminating immediately...")
        import os
        os._exit(0)
    self._shutdown_requested = True

In [59]:
def _attempt_recovery(self):
    logger.info("Attempting system recovery...")
    try:
        import gc
        gc.collect()
        if self.camera_manager:
            logger.info("Resetting camera connection...")
            self.camera_manager.release()
            time.sleep(2)
            self.camera_manager.initialize_camera()
        with self.web_lock:
            self.web_frame = None
        logger.info("Recovery attempt completed")
    except Exception as e:
        logger.error(f"Recovery failed: {e}")

In [60]:
def _cleanup(self):
    logger.info("Cleaning up resources...")
    if self.camera_manager:
        self.camera_manager.release()
    if self.detection_processor:
        stats = self.detection_processor.stats
        logger.info("Final statistics:")
        logger.info("  Total frames processed: %d", stats.get('frame_id', 0))
        logger.info("  Total detections: %d", stats.get('total_detections', 0))
        logger.info("  Asian hornets: %d", stats.get('total_velutina', 0))
        logger.info("  European hornets: %d", stats.get('total_crabro', 0))

## Interactive VespAI Application Setup
This cell demonstrates how to set up and run the VespAI application in a notebook, using variables to simulate command-line arguments (e.g., `--web`, `--resolution 720p`, `--motion`, `--conf 0.7`).

You can modify the variables below to control the application's configuration interactively.

In [61]:
# --- Set configuration variables as you would with CLI args ---
# Example: --web --resolution 720p --motion --conf 0.7

# Enable web dashboard
enable_web = True
# Set camera resolution (as string, e.g., '720p' or (1280, 720))
resolution = '1280x720'  # or '720p'
# Enable motion detection
enable_motion = True
# Set detection confidence threshold
confidence_threshold = 0.7
# Set video file (None to use camera)
video_file = None
# Save detections
save_detections = True
# Print detections to console
print_detections = True
# Frame delay (seconds)
frame_delay = 0.1
# Save directory
save_directory = 'monitor/detections'
# Additional config as needed

In [62]:
# --- Build a config dict to simulate CLI/config ---
config = {
    'enable_web': enable_web,
    'camera_resolution': resolution,
    'enable_motion': enable_motion,
    'confidence_threshold': confidence_threshold,
    'video_file': video_file,
    'save_detections': save_detections,
    'print_detections': print_detections,
    'frame_delay': frame_delay,
    'save_directory': save_directory,
    # Add more as needed
}

# Optionally, patch or monkeypatch the config loader to use this dict in the notebook.

In [68]:
import os

# --- Instantiate and run VespAI step by step ---
# Re-attach split methods to the class (notebook-safe)
for _name in [
    "initialize",
    "_initialize_camera",
    "_initialize_model",
    "_initialize_detection_processor",
    "_initialize_sms",
    "_initialize_web_interface",
    "_run_web_server",
    "run",
    "_handle_detection",
    "_save_detection_image",
    "_send_sms_alert",
    "_validate_initialization",
    "_signal_handler",
    "_attempt_recovery",
    "_cleanup",
]:
    if _name in globals() and callable(globals()[_name]):
        setattr(VespAIApplication, _name, globals()[_name])

app = VespAIApplication()

# Manually set config (simulate CLI/config)
app.config = type('Config', (), config)()

# Call initialization steps individually
# Notebook-safe config adapter so class methods expecting Config APIs still work
class NotebookConfig(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError as e:
            raise AttributeError(key) from e

    def __setattr__(self, key, value):
        self[key] = value

    def get_camera_resolution(self):
        value = self.get("camera_resolution", "1920x1080")
        if isinstance(value, tuple):
            return value
        if isinstance(value, str):
            presets = {"720p": (1280, 720), "1080p": (1920, 1080)}
            if value.lower() in presets:
                return presets[value.lower()]
            if "x" in value:
                w, h = value.lower().split("x", 1)
                return (int(w), int(h))
        return (1920, 1080)

    def get_web_config(self):
        host = self.get("web_host", "0.0.0.0")
        port = int(self.get("web_port", 5000))
        domain = self.get("domain_name", "localhost")
        public_url = self.get("public_url", f"http://{domain}:{port}")
        return {"host": host, "port": port, "public_url": public_url}

    def print_summary(self):
        logger.info("Notebook config loaded: %s", dict(self))

app.config = NotebookConfig(config)
app._initialize_camera()
# Ensure model_path is not None (avoids os.path/stat errors in model loading)
if not app.config.get("model_path"):
    app.config["model_path"] = "/opt/vespai/models/yolov5s-all-data.pt"

# Resolve model path with fallbacks before initializing model

base_dir = str(globals().get("project_root", os.getcwd()))
candidate_paths = [
    app.config.get("model_path"),
    "/opt/vespai/models/yolov5s-all-data.pt",
    os.path.join(base_dir, "models", "yolov5s-all-data.pt"),
    os.path.join(base_dir, "yolov5s.pt"),
    os.path.join(base_dir, "models", "yolov5s.pt"),
    "models/yolov5s-all-data.pt",
    "yolov5s.pt",
    "models/yolov5s.pt",
]

resolved_model_path = next(
    (p for p in candidate_paths if p and os.path.isfile(os.path.expanduser(p))),
    None,
)

if not resolved_model_path:
    raise RuntimeError(
        "Model file not found. Checked: " + ", ".join([p for p in candidate_paths if p])
    )

app.config["model_path"] = resolved_model_path
logger.info("Using model file: %s", resolved_model_path)
app._initialize_model()
app._initialize_detection_processor()
app._initialize_sms()
if app.config.enable_web:
    app._initialize_web_interface()

# Now you can run detection loop, or call app.run(), or step through further as needed
# Example: app.run()  # Uncomment to run the main loop (will block)
# Or step through frame processing, etc.

2026-03-06 08:43:44,606 - __main__ - INFO - Initializing camera...
2026-03-06 08:43:44,609 - src.vespai.core.detection - INFO - Initializing camera with resolution 1280x720
[ WARN:0@653.403] global cap.cpp:215 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name
[ WARN:0@653.404] global cap.cpp:215 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name
2026-03-06 08:43:44,614 - src.vespai.core.detection - INFO - Camera opened with device /dev/video25, backend 200
2026-03-06 08:43:44,622 - src.vespai.core.detection - INFO - Camera configured - Resolution: 1280x720, FPS: -1.0
[ WARN:0@653.416] global cap_v4l.cpp:1844 getProperty VIDEOIO(V4L2:/dev/video25): Unable to get camera FPS
2026-03-06 08:43:44,628 - src.vespai.core.detection - INFO - Camera initialized successfully
2026-03-06 08:43:45,131 - __main__ - INFO - Using model file: /home/sysadmin/vespai/models/yolov5s-all-data.pt
2026-03-06 08:43:45,132 - __main__ - INF

Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /home/sysadmin/.cache/torch/hub/master.zip


2026-03-06 08:43:51,308 - src.vespai.core.detection - INFO - Model classes: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'lapto

NameError: name '__file__' is not defined

In [ ]:
# --- Test USB camera with OpenCV ---
import cv2

# Try default USB camera (index 0)
cap = cv2.VideoCapture(0)
ret, frame = cap.read()
cap.release()

if ret and frame is not None:
    print("USB camera is working. Frame shape:", frame.shape)
else:
    print("USB camera not detected or not working. Try a different index or check connection.")

## USB camera override (Logitech C505e)
Use this to force VespAI to the working USB capture node before initialization.

In [ ]:
import os
import cv2

# Working capture node discovered from v4l2-ctl
USB_CAMERA_DEVICE = '/dev/video8'
os.environ['VESPAI_CAMERA_DEVICE'] = USB_CAMERA_DEVICE

# Quick verification
cap = cv2.VideoCapture(USB_CAMERA_DEVICE, cv2.CAP_V4L2)
ok, frame = cap.read()
cap.release()
print('device:', USB_CAMERA_DEVICE, 'ok:', ok, 'shape:', None if frame is None else frame.shape)

# Then run your app initialization cells as usual.

## Quick model switch
Edit `MODEL_CANDIDATES` order and run this cell to switch weights quickly. The first existing path is selected and exported as `VESPAI_MODEL_PATH` for the current notebook session.

In [ ]:
from pathlib import Path
import os

MODEL_CANDIDATES = [
    'models/hornet-best.pt',
    'models/yolov5s-all-data.pt',
    'models/yolov5s-official.pt',
    'yolov5s.pt',
]

selected_model = None
for candidate in MODEL_CANDIDATES:
    candidate_path = Path(candidate)
    if candidate_path.exists():
        selected_model = str(candidate_path.resolve())
        break

if selected_model is None:
    raise FileNotFoundError(f'No model file found. Checked: {MODEL_CANDIDATES}')

# VespAI config reads MODEL_PATH from environment
os.environ['MODEL_PATH'] = selected_model

# If an app object already exists in notebook state, update it directly too
if 'app' in globals() and getattr(app, 'config', None) is not None:
    try:
        app.config.config['model_path'] = selected_model
    except Exception:
        pass

print('Selected model:', selected_model)
print('Env set: MODEL_PATH')

# Optional generic fallback toggle during testing only:
# os.environ['VESPAI_ALLOW_GENERIC_MODEL'] = '1'